## Подключение

Ячейка подключения выполняется первой в каждом сеансе. Она подключает Google Drive (личная папка `practice2026` и общие данные группы `kg_data`), клонирует репозиторий группы в `/content/kg-practice-2026`, переключается на свою ветку `student-NN` с работой группы из `main` и регистрирует модули группы. В первых строках студент указывает свой номер, модуль и фамилию латиницей; токен GitHub лежит в секретах Colab под именем `GITHUB_TOKEN` (`ИНСТРУКЦИЯ_git.md`). Сдача этапа — последняя ячейка блокнота: `repo.submit(...)`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import sys, os, json, time, subprocess

STUDENT = 'student-07'                     # ← свой номер: вариант 7 — student-07
MY_MODULE = 'm07'                          # ← свой модуль: вариант 7 — m07
SURNAME = 'ivanov'                         # ← фамилия латиницей строчными буквами
REPO = 'github.com/<организация>/kg-practice-2026'     # репозиторий группы

KG = '/content/kg-practice-2026'                       # клон репозитория: код, модули, эталоны, сдача
DATA = '/content/drive/MyDrive/kg_data'                # общие данные группы на Drive (только чтение)
WORK = '/content/drive/MyDrive/practice2026'           # личная папка на Drive: свой набор и результаты
TEST = DATA + '/test_set'                              # общий тестовый набор
TEST_FACTS = WORK + '/test_facts'                      # свои прогоны модулей на тестовом наборе
ONTOLOGY = [KG + '/ontology/MocassinPlus_v2.ttl', KG + '/ontology/SciLibReco.ttl']
os.makedirs(WORK, exist_ok=True)

TOKEN = userdata.get('GITHUB_TOKEN')                   # секрет Colab с токеном GitHub
if not os.path.exists(KG):
    r = subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{REPO}.git', KG], capture_output=True, text=True)
    print(r.stderr.replace(TOKEN, '***'))
sys.path.insert(0, KG)
!pip install -q -r {KG}/requirements.txt
from mathkg.gitwork import Repo
repo = Repo(KG, STUDENT, MY_MODULE, SURNAME, token=TOKEN)
repo.start()                                           # своя ветка и всё, что уже принято в main
from mathkg import pipeline, registry
pipeline.setup(student_modules=KG + '/modules')
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
print('свой модуль:', MY_MODULE, my_tool)

# Этап 01. Корпус и base: свой набор через общую основу

Лекция 1 · сдать к встрече 3 · работа студента: 5–6 ч · вычисления в Colab на 1000 статей: base —
10–30 мин на CPU (измерено: 0,1–2 с на статью); распознавание выключных формул — 1–2 ч на GPU T4
фоном (не измерено, ориентир 0,2–0,5 с на формулу); базовые версии всех модулей на тестовом наборе —
секунды.

## Цель

Студент пропускает свой набор из 1000 статей через общую основу `base` пакета `mathkg`, получает для
каждой пригодной статьи представление `Doc` (текст, страницы, абзацы, предложения, отметки шрифта,
выключные формулы с номерами, гиперссылки PDF), отсеивает сканы и PDF со сломанной кодировкой,
запускает распознавание LaTeX выключных формул и измеряет время. Во второй части этапа студент
впервые видит свой модуль: запускает базовые версии всех десяти модулей на общем тестовом наборе и
разбирает, что находит базовая версия его модуля.

Этап не требует писать алгоритмы обработки PDF: их содержит `base`. Задача этапа — научиться
работать с общим кодом, понимать представление статьи, с которым будут работать все модули, и
знать свой набор в числах.

## Что прочитать

* SLP3, гл. 1 «Introduction» — задачи обработки языка и место извлечения информации среди них.
* Alammar, Grootendorst, гл. 1 «An Introduction to Large Language Models» — модели-кодировщики и
  генеративные модели, с которыми студент будет работать в модуле.
* Девлин, гл. 1 «Новая эра ИИ-агентов» и разд. 4.1 «Что такое граф знаний».
* Грейнджер, Тернбулл, Ирвин, гл. 1–2: поиск на основе ИИ и «миф о неструктурированных данных».
* `00_ПЛАН_практики.md` §1, §3–§6; `mathkg/README.md`; `02_МОДЕЛЬ_ДАННЫХ.md` §5 (тип `Doc`).

## Вход

* Свой набор: `practice2026/pdf/*.pdf` и `practice2026/index.csv` (поля `doc_id`, `file`), выданные
  руководителем.
* Репозиторий группы `kg-practice-2026` на GitHub: пакет `mathkg/`, онтологии `ontology/*.ttl`,
  блокноты этапов, папки сдачи; порядок работы с ним — `ИНСТРУКЦИЯ_git.md` в корне репозитория.
* Общие данные группы `kg_data/` на Google Drive (ярлык в «Моём диске», только чтение): общий
  тестовый набор `test_set/` (`pdf/`, `index.csv`, `base/` — уже обработан руководителем).
* Свой вариант: `варианты/Вариант_NN.md` (номер модуля NN).

## Шаги

### 1. Рабочая среда

Студент один раз готовит доступ к репозиторию группы (`ИНСТРУКЦИЯ_git.md`, раздел 1): принимает
приглашение в репозиторий, создаёт токен GitHub и кладёт его в секреты Colab под именем
`GITHUB_TOKEN`, добавляет в «Мой диск» ярлык на папку `kg_data`. Затем открывает блокнот этапа
`notebooks/stage01_Корпус_и_base.ipynb` из репозитория в Colab и сохраняет свою копию на Drive под
именем `stage01_<фамилия>.ipynb` («Файл → Сохранить копию на Диске»).

Первая ячейка блокнота — ячейка подключения; она одинакова во всех блокнотах практики, меняются
только три строки с номером студента, модулем и фамилией. Ячейка подключает Drive, клонирует
репозиторий группы в `/content/kg-practice-2026` (папка Colab, которая исчезает с концом сеанса),
переключается на свою ветку `student-NN` и подтягивает в неё всё, что уже принято в `main`:

```py
STUDENT = 'student-07'                     # ← свой номер: вариант 7 — student-07
MY_MODULE = 'm07'                          # ← свой модуль: вариант 7 — m07
SURNAME = 'ivanov'                         # ← фамилия латиницей строчными буквами

KG = '/content/kg-practice-2026'                       # клон репозитория: код, модули, эталоны, сдача
DATA = '/content/drive/MyDrive/kg_data'                # общие данные группы на Drive (только чтение)
WORK = '/content/drive/MyDrive/practice2026'           # личная папка на Drive: свой набор и результаты
TEST = DATA + '/test_set'                              # общий тестовый набор
TEST_FACTS = WORK + '/test_facts'                      # свои прогоны модулей на тестовом наборе
...
repo = Repo(KG, STUDENT, MY_MODULE, SURNAME, token=TOKEN)
repo.start()                                           # своя ветка и всё, что уже принято в main
```

Данные (свой набор, результаты base и модулей) живут на Drive и переживают конец сеанса. Код,
модуль, эталон, блокнот и отчёт сдаются в репозиторий последней ячейкой блокнота
`repo.submit(...)`: всё, что не отправлено на GitHub, пропадает вместе с папкой
`/content/kg-practice-2026`.

Проверка: `import mathkg; print(mathkg.__version__)` печатает номер версии. Если клонирование
не удалось, студент проверяет секрет `GITHUB_TOKEN` (доступ блокнота к секрету включён, токен не
просрочен) и строку `REPO`; если не открывается `TEST`, проверяет ярлык `kg_data` в «Моём диске».

### 2. Одна статья вручную

Перед обработкой всего набора студент разбирает одну статью: какие поля есть у `Doc` и что в них
лежит.

In [ ]:
from mathkg.base import load_doc, check_text_layer

def show_doc(doc, n_par=5):
    """Короткая сводка по Doc: язык, объём, число абзацев, формул, ссылок, отметок шрифта."""
    q = doc.meta['quality']
    print(doc.doc_id, doc.lang, 'стр.', doc.meta['n_pages'], 'символов', len(doc.text),
          'абзацев', len(doc.paragraphs), 'предложений', len(doc.sentences))
    print('формул', len(doc.formulas), 'с номером', sum(1 for f in doc.formulas if f.number),
          '| ссылок PDF', len(doc.links), '| сносок', len(doc.footnotes))
    print('качество слоя:', q, '→', check_text_layer(doc) or 'ok')
    for p in doc.paragraphs[:n_par]:
        print(f'  [стр. {p.page}]', doc.text[p.start:p.end][:120])

import csv
rows = list(csv.DictReader(open(WORK + '/index.csv', encoding='utf-8')))
r = rows[0]
doc = load_doc(f"{WORK}/pdf/{r['file']}", r['doc_id'])
show_doc(doc)

Затем студент смотрит отдельные поля:

* `doc.styled` — отрезки с отметками `bold`, `italic`, `math`: заголовки блоков «Теорема 2.1.»
  набраны полужирным, формулировки теорем в статьях из LaTeX — курсивом;
* `doc.formulas` — выключные формулы: текст из PDF (`text`), номер (`number`, например `'2.3'`),
  координаты (`bbox`) и пока пустое поле `latex`;
* `doc.links` — гиперссылки PDF с именованной целью (`target`): `cite.Diestel75` указывает на
  запись литературы, `equation.2.3` — на формулу, `theorem.2.1` — на теорему. У статей, свёрстанных
  Word, ссылок обычно нет.

In [ ]:
for s in doc.styled[:15]:
    print(s.kind, repr(doc.text[s.start:s.end][:60]))
for f in doc.formulas[:5]:
    print(f.number, repr(f.text[:80]))
for l in doc.links[:10]:
    print(l.target, repr(doc.text[l.start:l.end]) if l.start >= 0 else '—')

### 3. Весь набор

`build_corpus` обрабатывает все статьи из `index.csv`, пишет `practice2026/base/{doc_id}.json` и
сводку `base/docs.jsonl`. Уже обработанные статьи функция пропускает, поэтому после обрыва сеанса
ячейка запускается повторно.

In [ ]:
from mathkg.base import build_corpus
t0 = time.time()
stats = build_corpus(WORK + '/index.csv', WORK + '/pdf', WORK + '/base', log_every=100)
print(stats, f'{time.time() - t0:.0f} с')

Для пробы студент сначала запускает `build_corpus(..., limit=50)` и по времени на 50 статей
пересчитывает оценку для всего набора.

### 4. Сводка набора

In [ ]:
import pandas as pd
docs = pd.read_json(WORK + '/base/docs.jsonl', lines=True)
print(docs['status'].value_counts())
print(docs.groupby('lang')['n_pages'].describe())
print('время, с: среднее', docs['sec'].mean().round(2), 'макс', docs['sec'].max())
print(docs[docs.status != 'ok'][['doc_id', 'file', 'status', 'garbage_share']].head(20))

Статусы: `ok` — статья пригодна; `scan` — меньше 200 символов на страницу (скан без текстового
слоя); `broken_encoding` — больше 5 % нечитаемых символов (сломанная кодировка шрифтов); `error` —
PDF не открылся. Статьи со статусом, отличным от `ok`, дальше не обрабатываются; их список студент
передаёт руководителю, если их больше 3 % набора.

Студент открывает две-три статьи со статусом `broken_encoding` и смотрит текст: это нужно, чтобы
понимать, почему они отсеяны. Скан с распознанным текстовым слоем проходит проверку, но содержит шум
(«[0, l]2 —•»); такие статьи студент отмечает в отчёте.

### 5. Распознавание формул

Текстовый слой PDF не хранит структуру формулы: индексы попадают в строку, дроби распадаются.
`base.recognize_formulas` вырезает каждую выключную формулу по координатам из изображения страницы
и передаёт её модели pix2tex, которая возвращает LaTeX. Студент только запускает функцию: модель
работает на GPU, поэтому блокнот переключается на среду с GPU T4.

In [ ]:
!pip install -q pix2tex

In [ ]:
from mathkg.base import read_doc, save_doc, recognize_formulas
from pix2tex.cli import LatexOCR
ocr = LatexOCR()                                   # модель загружается один раз

ok_ids = docs[docs.status == 'ok'].doc_id.tolist()
files = dict(zip(docs.doc_id, docs.file))

def recognize_all(ids, log_every=20):
    """Распознаёт формулы статей ids и сохраняет Doc. Статьи, где LaTeX уже есть, пропускает."""
    done = 0
    for k, doc_id in enumerate(ids, 1):
        doc = read_doc(WORK + '/base', doc_id)
        if not doc.formulas or all(f.latex for f in doc.formulas):
            continue
        recognize_formulas(doc, f"{WORK}/pdf/{files[doc_id]}", model=ocr)
        save_doc(doc, WORK + '/base')
        done += 1
        if k % log_every == 0:
            print(k, 'статей')
    return done

t0 = time.time(); recognize_all(ok_ids[:5]); sec = time.time() - t0
n = sum(len(read_doc(WORK + '/base', i).formulas) for i in ok_ids[:5])
print(f'{n} формул за {sec:.0f} с → {sec / max(1, n):.2f} с на формулу')

По времени на пять статей студент оценивает полный прогон и запускает `recognize_all(ok_ids)`.
Функция сохраняет каждую статью сразу после распознавания, поэтому её можно запускать частями в
разные дни. Затем студент сравнивает для трёх формул текст из PDF и LaTeX:

In [ ]:
from mathkg.base import read_doc
d = read_doc(WORK + '/base', ok_ids[0])
for f in d.formulas[:3]:
    print(f.number, '| PDF:', f.text[:60], '| LaTeX:', f.latex)

### 6. Свой модуль на тестовом наборе

Общий тестовый набор (20–30 статей) уже обработан base руководителем. Студент запускает базовые
версии всех десяти модулей и смотрит результат своего.

In [ ]:
from mathkg import pipeline, registry
pipeline.setup()                                   # базовые версии m01–m10
print(registry.extraction_order())
st = pipeline.run_corpus(TEST + '/base', TEST_FACTS)
for name, s in st.items():
    print(name, s)

In [ ]:
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
spec = registry.REGISTRY[my_tool]
print(my_tool, '| зависит от:', spec.requires, '| слой:', spec.layer)
print(spec.description)

from mathkg.pipeline import load_facts
from mathkg.base import read_doc
test_ids = pd.read_json(TEST + '/base/docs.jsonl', lines=True).query("status == 'ok'").doc_id.tolist()
d = read_doc(TEST + '/base', test_ids[0])
f = load_facts(TEST_FACTS, my_tool, test_ids[0])
for x in f.fragments[:10]:
    print(x.id, x.cls, x.number, x.confidence, '|', d.text[x.start:x.end][:90])
for r in f.relations[:10]:
    print(r.subj, r.pred, r.obj)

Студент читает исходный код базовой версии своего модуля (`mathkg/modules/mNN_*.py` в репозитории)
и записывает в отчёт, на чём держится каждое правило и на каких статьях тестового набора оно
ошибается (три примера пропусков и три примера ложных находок).

## Выход

* `practice2026/base/{doc_id}.json` — `Doc` каждой пригодной статьи (с LaTeX формул после шага 5);
* `practice2026/base/docs.jsonl` — сводка набора;
* `practice2026/test_facts/` — результаты базовых версий модулей на тестовом наборе;
* `practice2026/stage01/REPORT.md` (ячейка сдачи копирует его в `students/NN_фамилия/stage01/`).

## Самопроверка

* Статей со статусом `ok` — не меньше 95 % набора; иначе список отсеянных передан руководителю.
* Среднее время base — доли секунды на статью; несколько секунд на статью говорят о том, что
  блокнот работает не с локальной копией и каждый PDF читается по сети (копирование `pdf/` в
  `/content` ускоряет обработку).
* У статей из LaTeX есть гиперссылки с целями `cite.` и отметки `italic`; у статей из Word ссылок
  нет, отметки `bold` есть.
* Доля выключных формул с номером — заметная (обычно от четверти до половины).
* После распознавания у большинства выключных формул поле `latex` заполнено.

## Что сдать

Сдача — ячейка сдачи в конце блокнота и запрос на слияние «Этап 1: фамилия» в `main`
(`ИНСТРУКЦИЯ_git.md`, раздел 4). В репозиторий попадают блокнот
`students/NN_фамилия/stage01.ipynb` с выводами и папка `students/NN_фамилия/stage01/` с отчётом
`REPORT.md` и небольшими результатами этапа. В отчёте: таблица статусов набора, языки, число
страниц, время base и распознавания формул (на статью и на весь набор), пример одной статьи (первые
абзацы, отметки шрифта, три формулы с LaTeX), разбор базовой версии своего модуля с примерами
ошибок.

## Как результат станет инструментом агента

`read_doc(WORK + '/base', doc_id)` — источник для инструмента `get_article_text(doc_id)` этапа 11:
агент получает текст статьи и её абзацы из `Doc`. Базовая версия своего модуля с первого дня уже
является инструментом: `registry.agent_tools` превращает каждый модуль в функцию с параметром
`doc_id` и описанием из docstring.

## Частые ошибки

* Папка `kg_data` открыта по ссылке, но ярлык в «Мой диск» не добавлен: путь `TEST` не существует.
* Работа сохранена только в `/content/kg-practice-2026` и не сдана `repo.submit(...)`: с концом
  сеанса Colab она пропадает.
* Распознавание формул запущено в среде без GPU: модель работает в десятки раз медленнее.
* После распознавания `Doc` не сохранён `save_doc`, и LaTeX теряется при следующем чтении.
* Результаты модулей на тестовом наборе записаны в `kg_data/test_set/`: эта папка только для
  чтения, свои результаты пишутся в `practice2026/test_facts/`.
* В отчёте отсеянные статьи не перечислены, хотя их больше 3 %.

## Вопросы к защите

1. Чем текстовый слой PDF отличается от изображения страницы и почему по нему нельзя восстановить
   формулу?
2. Что хранит `Doc.styled` и как отметки шрифта помогут найти заголовки теорем?
3. Что такое именованная цель гиперссылки PDF и откуда она берётся у статей из LaTeX?
4. Почему все модули работают с одним общим `Doc`, полученным base?
5. Как `build_corpus` продолжает работу после обрыва сеанса?
6. Какие статьи отсеивает `check_text_layer` и какой шум она не замечает?
7. От каких модулей зависит ваш модуль и что произойдёт с его результатом, если модуль-зависимость
   ошибётся?

**Вариант.** На шаге 6 студент разбирает базовую версию модуля своего варианта: что она находит, по
каким признакам и где ошибается. Особенности, на которые стоит обратить внимание уже на этом этапе,
перечислены в разделе «Этап 1» файла `варианты/Вариант_NN.md`.

## Сдача этапа

Перед сдачей блокнот сохраняют (Ctrl+S), а отчёт `REPORT.md` кладут в папку этапа на Drive (`practice2026/stageNN/`). Ячейка копирует в свою папку сдачи `students/NN_фамилия/` блокнот и небольшие файлы этапа (отчёт, таблицы, запросы, журналы), отправляет свои файлы в ветку `student-NN` и печатает ссылку на запрос на слияние в `main`; запрос открывают по ссылке с заголовком «Этап N: фамилия».

In [ ]:
STAGE = 1
repo.save_notebook(f'stage{STAGE:02d}_{SURNAME}.ipynb', STAGE)   # имя своей копии блокнота на Drive
repo.collect(f'{WORK}/stage{STAGE:02d}', STAGE)                  # отчёт и небольшие результаты этапа
repo.submit(f'Этап {STAGE}: {SURNAME}')